# Training Series: Introduction to eXtended Discontinuous Galerkin Methods for Multi-Phase Flow Problems <br> - Hands-On Worksheets

In [1]:
#r "../BoSSSpad/BoSSSpad.dll"
//#r "C:\Program Files (x86)\FDY\BoSSS\bin\Release\net6.0\BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.LinSolvers;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Platform.LinAlg;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.Grid.RefElements;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

# Hands-On 4: Steady-state Stokes equations within a two-phase setting

## Problem statement

The Stokes equations are given as

$$ -\frac{1}{Re} \Delta \vec{u}
 + \nabla p
     \  = \vec{g} $$

$$\text{div} (\vec{u})
      = 0, $$

where $Re \in \mathbb{R} $ denotes the Reynolds number. We consider two types of boundary conditions for the Stokes equation,
Dirichlet (on $\Gamma_D \subset \Omega$) and Neumman (on $\Gamma_N  \subset \Omega$). Those are defined as

$$ \vec{u}  =\vec{u}_D 
         \  \text{ on } \Gamma_D\ \text{ (Dirichlet)}, \\
        %
        \left( -\frac{1}{Re}\ \nabla  \vec{u} +  I p \right) 
        \vec{n}\vert_{\delta \Omega}         = 0 
                \  \text{ on } \Gamma_N\ \text{ (Neumann) } .$$

To indicate at which point of the boundary which condition is valid, 
i.e. wether a certain point is either Dirichlet or Neumann we
define **IsDirichletBndy** which defines a mapping 

$$ \vec{x} \mapsto \{ \text{true}, \text{false} \}, $$

where **true** actually indicates a Dirichlet boundary.
Since this function is defined as a global delegate, it can be altered later on. 
In the same manner, the function **UDiri** defines the Dirichlet-value for the velocity at the boundary.

In [2]:
static class BndyMap { 

    public static Func<double[],bool> IsDirichletBndy = null;

    public static Func<double[],double[]> UDiri = null;

}

## Velocity divergence and pressure gradient

At first, we implement the velocity divergence, 
i.e. the continuity equation. We use the strong form:

$$ b(\vec{u},v) = 
   \oint_{(\Gamma \backslash \Gamma_D) \cup \mathfrak{I}} 
          \{v\} \llbracket \vec{u} \rrbracket \cdot \vec{n}_\Gamma dA 
   - \int_{\Omega} \text{div}(\vec{u}) \ v dV$$

In [3]:
public class Divergence : 
        BoSSS.Foundation.IEdgeForm, // edge integrals
        BoSSS.Foundation.IVolumeForm     // volume integrals
{
    /// The parameter list for the divergence is empty:
    public IList<string> ParameterOrdering { 
        get { return null; } 
    }
 
    /// We have a vector argument variable, 
    /// the velocity [ u, v ] = u
    /// (our trial function):
    public IList<String> ArgumentOrdering { 
        get { return new string[] { "u", "v" }; } 
    }
 
    public TermActivationFlags VolTerms {
        get {
            return TermActivationFlags.AllOn;
        }
    }
 
    public TermActivationFlags InnerEdgeTerms {
        get {
            return TermActivationFlags.AllOn; 
        }
    }
 
    public TermActivationFlags BoundaryEdgeTerms {
       get {
           return TermActivationFlags.AllOn;
        }
    }
 
    /// In the volume part, the integrand is div(u)*v :
    public double VolumeForm(ref CommonParamsVol cpv, 
        double[] U, double[,] GradU, 
        double V, double[] GradV) {

        double Acc = 0;
        // == TODO == add the volume term
        for(int d = 0; d < cpv.D; d++) {
            Acc -= GradU[d,d]*V;
        }
        return Acc;
    }
 
    /// On interior cell boundaries, we use a velocity penalty,
    /// $\mean{v} \jump{u} \cdot n_\Gamma$:
    public double InnerEdgeForm(ref CommonParams inp, 
        double[] U_IN, double[] U_OT, double[,] GradU_IN, double[,] GradU_OT, 
        double V_IN, double V_OT, double[] GradV_IN, double[] GradV_OT) {
 
        double Acc = 0;
        // == TODO == add the inner edge term of the strong div form
        for(int d = 0; d < inp.D; d++) {
            Acc += 0.5*(V_IN + V_OT)*(U_IN[d] - U_OT[d])*inp.Normal[d];
        }
        return Acc;
    }
 
    /// On the domain boundary, we have to distinguish between 
    /// Dirichlet- and Neumann-boundary conditions; the function
    /// \code{uDiri} defines which of the two actually applies:
    public double BoundaryEdgeForm(ref CommonParamsBnd inp, 
        double[] U_IN, double[,] GradU_IN, double V_IN, double[] GradV_OT) {
 
        double Acc = 0;
        // == TODO == add the boundary edge term of the strong div form. Hint: enforce a Dirichlet B.C. via \code{BndyMap.UDiri} (gNeu = 0.0)    
        if(BndyMap.IsDirichletBndy(inp.X)) {
            // On the Dirichlet boundary, the outer value for the velocity
            // is given by the function/delegate 'UDiri':
            double[] UD = BndyMap.UDiri(inp.X);
            for(int d = 0; d < inp.D; d++) {
                Acc += (U_IN[d] - UD[d])*inp.Normal[d]*V_IN;
            }
        } else {
            // On the Neumann boundary, we do not know an outer value for the
            // velocity, so there is no penalization at all:
            Acc = 0;    
        }
        return Acc;
    }
}

In [4]:
public class Divergence_Interface : 
                    ILevelSetForm  // interface integrals
{
    /// We do not use parameters (e.g. variable viscosity, ...)
    /// at this point: so this can be null
    public IList<string> ParameterOrdering { 
        get { return new string[0]; } 
    } 
    /// but we have one argument variable, $u$ (our trial function)
    public IList<String> ArgumentOrdering { 
        get { return new string[] { "u", "v" }; } 
    }

    public TermActivationFlags LevelSetTerms {
        get {
            return TermActivationFlags.UxV;
        }
    }

    public int LevelSetIndex {
        get { return 0; }
    }

    public string NegativeSpecies {
        get { return "A"; }
    }

    public string PositiveSpecies {
        get { return "B"; }
    }

    /// The integrand for the integral on the interface (inner edge),
    public virtual double InnerEdgeForm(ref CommonParams inp, 
        double[] uA, double[] uB, double[,] Grad_uA, double[,] Grad_uB,
        double vA, double vB, double[] Grad_vA, double[] Grad_vB) {
 
        double Acc = 0;
        // == TODO == add the inner edge term of the strong div form
        for(int d = 0; d < inp.D; d++) {
            Acc += 0.5*(vA + vB)*(uA[d] - uB[d])*inp.Normal[d];
        }
        return Acc;
    } 
    
}

## The gradient-operator

For the variational formulation of the gradient operator, a vector-valued
test-function is required. Unfourtunately, this is not supported by 
*BoSSS*. Therefore we have to discretize the gradient component-wise,
i.e. as $\partial_{x}$ and $\partial_y$. 

A single derivative can obviously be expressed as a divergence by the
identity $ \partial_{x_d} = \text{div}( \vec{e}_d p ) $.

In [5]:
class Gradient_d : 
        BoSSS.Foundation.IEdgeForm, // edge integrals 
        BoSSS.Foundation.IVolumeForm     // volume integrals 
{ 
    public Gradient_d(int _d) { 
        this.m_d = _d; 
    } 
 
    /// The component index of the gradient:
    int m_d; 
 
    /// As ususal, we do not use parameters:
    public IList<string> ParameterOrdering {  
        get { return null; }  
    } 
 
    /// We have one argument $u$:
    public IList<String> ArgumentOrdering {  
        get { return new string[] { "p" }; }  
    } 
 
    public TermActivationFlags VolTerms { 
        get { return TermActivationFlags.AllOn; } 
    } 
 
    public TermActivationFlags InnerEdgeTerms { 
        get { return (TermActivationFlags.AllOn); } 
    } 
 
    public TermActivationFlags BoundaryEdgeTerms { 
       get { return TermActivationFlags.AllOn; } 
    } 
 
    /// The volume integrand, for a vector-valued test-function $\vec{v}$
    /// would be $-\operatorname{div}{\vec{v}} p$. Our test function $v$
    /// is scalar-valued, so e.g. for $\code{d} = 0$ we have $\vec{v} = (v,0)$. 
    /// In this case, our volume integrand reduces as 
    /// $-\operatorname{div}{\vec{v}} \psi = -\partial_x v \psi$:
    public double VolumeForm(ref CommonParamsVol cpv,  
           double[] P, double[,] GradP,  
           double V, double[] GradV) { 
 
        double Acc = 0; 
        // == TODO == add the volume form
        Acc -= P[0]*GradV[m_d]; 
        return Acc; 
    }         
 
    /// On interior cell edges, we simply use a central-difference flux.
    /// Again, we consider a scalar test function, so we have
    /// $ \jump{p} \vec{v} \cdot \vec{n} = \jump{p} v n_d $,
    /// where $n_d$ is the $d$--th component of $\vec{n}$:
    public double InnerEdgeForm(ref CommonParams inp,  
        double[] P_IN, double[] P_OT, double[,] GradP_IN, double[,] GradP_OT,  
        double V_IN, double V_OT, double[] GradV_IN, double[] GradV_OT) { 
 
        double Acc = 0; 
        // == TODO == add the inner edge form
        Acc += 0.5*(P_IN[0] + P_OT[0])*inp.Normal[m_d]*(V_IN - V_OT); 
        return Acc;   
     } 
 
    public double BoundaryEdgeForm(ref CommonParamsBnd inp,  
        double[] P_IN, double[,] GradP_IN, double V_IN, double[] GradV_OT) { 
 
        double Acc = 0; 
        // == TODO == add the boundary edge form. Hint: the total stress at the Neumann boundary should be zero!
        if(BndyMap.IsDirichletBndy(inp.X)) {
            // On the Dirichlet boundary, we do not know an outer value for 
            // the pressure, so we have to take the inner value:
            Acc += P_IN[0]*inp.Normal[m_d]*V_IN;
        } else {
            // On the Neumann boundary, we want the total stress to be zero,
            // so there is no contribution from the pressure:
            Acc = 0;
        } 
        return Acc;               
    } 
}

In [6]:
public class Gradient_d_Interface : 
                    ILevelSetForm  // interface integrals
{
    public Gradient_d_Interface(int _d) { 
        this.m_d = _d; 
    } 
 
    /// The component index of the gradient:
    int m_d; 
    
    /// We do not use parameters (e.g. variable viscosity, ...)
    /// at this point: so this can be null
    public IList<string> ParameterOrdering { 
        get { return new string[0]; } 
    } 
    /// but we have one argument variable, $u$ (our trial function)
    public IList<String> ArgumentOrdering { 
        get { return new string[] { "p" }; } 
    }

    public TermActivationFlags LevelSetTerms {
        get {
            return TermActivationFlags.UxV;
        }
    }

    public int LevelSetIndex {
        get { return 0; }
    }

    public string NegativeSpecies {
        get { return "A"; }
    }

    public string PositiveSpecies {
        get { return "B"; }
    }

    /// The integrand for the integral on the interface (inner edge)
    public virtual double InnerEdgeForm(ref CommonParams inp, 
        double[] uA, double[] uB, double[,] Grad_uA, double[,] Grad_uB,
        double vA, double vB, double[] Grad_vA, double[] Grad_vB) {
 
        double Acc = 0; 
        // == TODO == add the inner edge form
        Acc += 0.5*(uA[0] + uB[0])*inp.Normal[m_d]*(vA - vB); 
        return Acc; 
    } 
    
}

## Tests on pressure gradient and velocity divergence

If our implementation is correct, we created a discretization of 

$$ \left[ \begin{array}{cc}
    0                 & \nabla p \\
  -\operatorname{div}(\vec{u}) & 0      \\
 \end{array} \right]$$

so the matrix should have the form 

$$\left[ \begin{array}{cc}
    0     & B      \\
    B^T   & 0      \\
 \end{array} \right]
 =: M,$$
 
i.e. $M$ should be symmetric.
We are testing this using a channel flow configuration using an equidistant grid.
- Domain: $\Omega    :=  (0,10) \times (-1,1)$
- Neumann boundary (i.e. the outlet of the channel on the right): $\Gamma_N  :=  \{ (x,y) | \ x = 10 \}$
- Dirichlet boundary (i.e. inlet and walls): $   \Gamma_D  :=  \delta \Omega \setminus \Gamma_N$
- Velocity boundary value at inlet and walls: $  \vec{u}_D :=  (1 - y^2, 0)$

We create a grid, a DG basis for velocity and pressure 
and a variable mapping

In [7]:
var xNodesChannel = GenericBlas.Linspace(0,10,31);// 30 cells in x-direction
var yNodesChannel = GenericBlas.Linspace(-1,1,7); // 6 cells in y-direction
var grdChannel    = Grid2D.Cartesian2DGrid(xNodesChannel, yNodesChannel);

In [8]:
LevelSet LevelSetField = new LevelSet(new Basis(grdChannel, 1), "Interface");   // line interface
LevelSetField.ProjectField((double[] X) => (X[0]- 5.0)/5.0 - X[1]);
LevelSetTracker LsTrk = new LevelSetTracker(grdChannel.GridData, XQuadFactoryHelper.MomentFittingVariants.Saye, 1, new string[] {"A", "B"}, LevelSetField);
LsTrk.UpdateTracker(0.0);

In [9]:
var VelBChannel   = new XDGBasis(LsTrk, 2);  // velocity basis k=2
var PBChannel   = new XDGBasis(LsTrk, 1);  // pressure basis l=k-1
var varMapChannel = new UnsetteledCoordinateMapping(
                       VelBChannel, VelBChannel, PBChannel); // variable mapping

We specify the boundary conditions as delegates:

In [10]:
Func<double[],bool> IsDirichletBndy_Channel 
        = (X => Math.Abs(X[0] - 10) > 1.0e-10); // it's Dirichlet, if x != 10
Func<double[],double[]> UDiri_Channel 
        = (X => new double[2] { 1.0 - X[1]*X[1], 0});

Let's create the operator which contains only the pressure gradient
and velocity divergence:

In [11]:
XDifferentialOperatorMk2 GradDivXDG = new XDifferentialOperatorMk2(3,3, // 3 vars. in dom. & codom.
    QuadOrderFunc.Linear(), // linear operator
    LsTrk.SpeciesNames, // names of species
    "u", "v", "p",  // names of domain variables
    "mom_x", "mom_y", "conti"); // names of codom. vars
GradDivXDG.EquationComponents["mom_x"].Add(new Gradient_d(0)); 
GradDivXDG.EquationComponents["mom_x"].Add(new Gradient_d_Interface(0)); 
GradDivXDG.EquationComponents["mom_y"].Add(new Gradient_d(1)); 
GradDivXDG.EquationComponents["mom_y"].Add(new Gradient_d_Interface(1)); 
GradDivXDG.EquationComponents["conti"].Add(new Divergence());
GradDivXDG.EquationComponents["conti"].Add(new Divergence_Interface());
GradDivXDG.Commit();

We create the matrix of the **GradDiv**-operator for 
the channel configuration. Before that, we have to set values for the 
global **IsDirichletBndy** and **UDiri**-variables.

In [12]:
BndyMap.IsDirichletBndy  = IsDirichletBndy_Channel;
BndyMap.UDiri             = UDiri_Channel;

BlockMsrMatrix GradDivXMatrix_Channel = new BlockMsrMatrix(varMapChannel);
double[] Affine = new double[varMapChannel.LocalLength];
GradDivXDG.GetMatrixBuilder(LsTrk, varMapChannel, null, varMapChannel).ComputeMatrix(GradDivXMatrix_Channel, Affine);

Finally, we can test the symmetry of the matrix:

In [13]:
var ErrMtx = GradDivXMatrix_Channel - GradDivXMatrix_Channel.Transpose();
ErrMtx.InfNorm()

1.2878587085651816E-14

## Adding the viscous operator, forming the Stokes operator

We use the SIP-operator from the previous chapter **SIP** to model the viscous terms:

In [14]:
/// We implement Reynolds number as global, static variables.
static class ProblemMap { 

    public static double ReA = 20.0;
    public static double ReB = 20.0;
}

In [15]:
public class Viscous : 
        IEdgeForm,   // edge integrals
        IVolumeForm, // volume integrals
        IEquationComponentCoefficient, // update of coefficients required for penalty parameters 
        IEquationComponentSpeciesNotification   // switch coefficients based on species
{
    /// The velocity component:
    int m_d;
 
    public Viscous(int _d) {
        this.m_d = _d;    
    }


    public void SetParameter(string speciesName) {
        switch (speciesName) {
            case "A": ReSpc = ProblemMap.ReA; break;
            case "B": ReSpc = ProblemMap.ReB; break;
            default: throw new ArgumentException("Unknown species.");
        }
    }

    double ReSpc;

 
    /// We do not use parameters:
    public IList<string> ParameterOrdering { 
        get { return new string[0]; } 
    }
 
    /// Depending on \code{d}, the argument variable
    /// should be either $u$ or $v$:
    public IList<String> ArgumentOrdering { 
        get { 
            switch(m_d) {
                case 0  : return new string[] { "u" }; 
                case 1  : return new string[] { "v" }; 
                default : throw new Exception();
            }
        } 
    }
 
    /// The \code{TermActivationFlags}, as usual:
    public TermActivationFlags VolTerms {
        get {
            return TermActivationFlags.GradUxGradV;
        }
    }
 
    public TermActivationFlags InnerEdgeTerms {
        get {
            return TermActivationFlags.AllOn;
        }
    }
 
    public TermActivationFlags BoundaryEdgeTerms {
       get {
           return TermActivationFlags.AllOn;
        }
    }
 
    /// The integrand for the volume integral:
    public double VolumeForm(ref CommonParamsVol cpv, 
           double[] U, double[,] GradU,
           double V, double[] GradV) {               
        double acc = 0;
        for(int d = 0; d < cpv.D; d++)
            acc += GradU[0, d] * GradV[d];
        return (1/ReSpc)*acc;
    }
 
 
    /// The integrand for the integral on the inner edges:
    public double InnerEdgeForm(ref CommonParams inp, 
        double[] U_IN, double[] U_OT, double[,] GradU_IN, double[,] GradU_OT, 
        double V_IN, double V_OT, double[] GradV_IN, double[] GradV_OT) {
 
        double eta = PenaltyFactor(inp.jCellIn, inp.jCellOut);
 
        double Acc = 0.0;
        for(int d = 0; d < inp.D; d++) { // loop over vector components 
            // consistency term: -({{ \/u }} [[ v ]])*Normal
            // index d: spatial direction
            Acc -= 0.5 * (GradU_IN[0, d] + GradU_OT[0, d])*(V_IN - V_OT)
                       * inp.Normal[d];
 
            // the symmetry term -({{ \/v }} [[ u ]])*Normal
            Acc -= 0.5 * (GradV_IN[d] + GradV_OT[d])*(U_IN[0] - U_OT[0])
                       * inp.Normal[d];;
        }
 
        // the penalty term eta*[[u]]*[[v]]
        Acc += eta*(U_IN[0] - U_OT[0])*(V_IN - V_OT);
        return (1/ReSpc)*Acc;
 
    }
 
    /// The integrand on boundary edges, i.e. on $\partial \Omega$:
    public double BoundaryEdgeForm(ref CommonParamsBnd inp, 
        double[] U_IN, double[,] GradU_IN, double V_IN, double[] GradV_IN) {
 
 
        double Acc = 0.0;
 
        if(BndyMap.IsDirichletBndy(inp.X)) {
            // Dirichlet boundary conditions
            double uBnd = BndyMap.UDiri(inp.X)[m_d];
 
            for(int d = 0; d < inp.D; d++) { // loop over vector components 
                // consistency term:
                Acc -= (GradU_IN[0, d])*(V_IN) * inp.Normal[d];
                // symmetry term:
                Acc -= (GradV_IN[d])*(U_IN[0]- uBnd) * inp.Normal[d];
            }
 
            // penalty term
            double eta = PenaltyFactor(inp.jCellIn, -1);
            Acc += eta*(U_IN[0] - uBnd)*(V_IN);
            
        } else {
            // Neumann boundary conditions, i.e. zero-stress:
            Acc = 0;
        }
 
        return (1/ReSpc)*Acc;
    }
            
    MultidimensionalArray cj;
    double penalty_base;
            
    double PenaltyFactor(int jCellIn, int jCellOut) {
        double PenaltySafety = 2;
        double cj_in         = cj[jCellIn];
        double eta           = penalty_base * cj_in * PenaltySafety;
        if(jCellOut >= 0) {
            double cj_out = cj[jCellOut];
            eta           = Math.Max(eta, penalty_base * cj_out * PenaltySafety);
        }
        return eta;
    }
            
            
    /// Update of penalty length scales.
    public void CoefficientUpdate(CoefficientSet cs, int[] DomainDGdeg, int TestDGdeg) {
        int D = cs.GrdDat.SpatialDimension;
        double _D = D;
        double _p = DomainDGdeg.Max();

        double penalty_deg_tri = (_p + 1) * (_p + _D) / _D; // formula for triangles/tetras
        double penalty_deg_sqr = (_p + 1.0) * (_p + 1.0); // formula for squares/cubes

        penalty_base = Math.Max(penalty_deg_tri, penalty_deg_sqr); // the conservative choice

        cj = ((GridData)(cs.GrdDat)).Cells.cj;
    }
}

In [16]:
public class Viscous_Interface : 
                    ILevelSetForm,  // interface integrals
                    ILevelSetEquationComponentCoefficient  // update of coefficients (e.g. length scales) required for penalty parameters 
{
    /// The velocity component:
    int m_d;
 
    public Viscous_Interface(int _d) {
        this.m_d = _d;    
    }
 

    /// We do not use parameters (e.g. variable viscosity, ...)
    /// at this point: so this can be null
    public IList<string> ParameterOrdering { 
        get { return new string[0]; } 
    } 
    /// but we have one argument variable, $u$ (our trial function)
    public IList<String> ArgumentOrdering { 
        get { return new string[] { "u", "v" }; } 
    }

    public TermActivationFlags LevelSetTerms {
        get {
            return TermActivationFlags.UxV | TermActivationFlags.GradUxV | TermActivationFlags.UxGradV;
        }
    }

    public int LevelSetIndex {
        get { return 0; }
    }

    public string NegativeSpecies {
        get { return "A"; }
    }

    public string PositiveSpecies {
        get { return "B"; }
    }

    /// For the computation of the penalty factor $\eta$,
    /// we require some length scale for each cell and 
    /// the polynomial degree of the DG approximation.
    MultidimensionalArray cjA;
    MultidimensionalArray cjB;

    double penalty_base; // base factor must be scaled by polynomial degree  

    public void CoefficientUpdate(CoefficientSet csA, CoefficientSet csB, int[] DomainDGdeg, int TestDGdeg) {
        cjA = ((GridData)(csA.GrdDat)).Cells.cj; //csA.CellLengthScales;
        cjB = ((GridData)(csB.GrdDat)).Cells.cj; //csB.CellLengthScales;

        double _p = DomainDGdeg.Max();
        double _D = csA.GrdDat.SpatialDimension;
        double penalty_deg_tri = (_p + 1) * (_p + _D) / _D; // formula for triangles/tetras
        double penalty_deg_sqr = (_p + 1.0) * (_p + 1.0); // formula for squares/cubes
            
        penalty_base = Math.Max(penalty_deg_tri, penalty_deg_sqr);
    }   
    
            
    /// The safety factor for the penalty factor should be in the order of 1.
    /// A very large penalty factor increases the condition number of the system, but without affecting stability.
    /// A very small penalty factor yields to an unstable discretization.
    public double PenaltySafety = 2.2; 

    /// The actual computation of the penalty factor, which should be used in the \code{InnerEdgeForm} and \code{BoundaryEdgeForm} functions.
    /// Hint: for the parameters \code{jCellIn}, \code{jCellOut} and \code{g}, take a look at \code{CommonParams} and \code{CommonParamsBnd}.
    double PenaltyFactor(int jCellIn, int jCellOut) {
        double penaltySizeFactor_A = cjA[jCellIn];
        double penaltySizeFactor_B = cjB[jCellOut];

        double penaltySizeFactor = Math.Max(penaltySizeFactor_A, penaltySizeFactor_B);

        double scaledPenalty = penaltySizeFactor * PenaltySafety * penalty_base;
        if(scaledPenalty.IsNaNorInf())
            throw new ArithmeticException("NaN/Inf detected for penalty parameter.");
        return scaledPenalty;
    }    

    /// The integrand for the integral on the interface (inner edge),
    /// 
    ///   -( {\nabla u} [[v]]) \cdot \vec{n}_{\Gamma} 
    ///   -( {\nabla v} [[u]]) \cdot \vec{n}_{\Gamma} 
    ///   + \eta [[u]] [[v]] :
    /// 
    public virtual double InnerEdgeForm(ref CommonParams inp, 
        double[] uA, double[] uB, double[,] Grad_uA, double[,] Grad_uB,
        double vA, double vB, double[] Grad_vA, double[] Grad_vB) {
 
        double eta = PenaltyFactor(inp.jCellIn, inp.jCellOut);
        double ReA = ProblemMap.ReA;
        double ReB = ProblemMap.ReB;    
 
        double Acc = 0.0;

        for(int d = 0; d < inp.D; d++) { // loop over vector components 
            // consistency term: -({ \/u } [[ v ]])*Normal
            // index d: spatial direction
            Acc -= 0.5 * ((1.0/ReA) * Grad_uA[m_d, d] + (1.0/ReB) * Grad_uB[m_d, d]) * (vA - vB) * inp.Normal[d];
 
            // symmetry term: -({ \/v } [[ u ]])*Normal
            Acc -= 0.5 * ((1.0/ReA) * Grad_vA[d] + (1.0/ReB) * Grad_vB[d]) * (uA[m_d] - uB[m_d]) * inp.Normal[d];
        }
 
        // penalty term: eta*[[u]]*[[v]]
        double mufactor = (Math.Abs(ReA) > Math.Abs(ReB)) ? (1.0/ReB) : (1.0/ReA);
        Acc += mufactor * eta * (uA[m_d] - uB[m_d]) * (vA - vB);        

        return Acc;
    } 
    
}

Finally, we are ready to implement the Stokes operator:

In [17]:
XDifferentialOperatorMk2 xdgStokes = new XDifferentialOperatorMk2(3, 3, // 3 vars. in dom. & codom.
    QuadOrderFunc.Linear(), // linear operator
    LsTrk.SpeciesNames, // names of species
    "u", "v", "p",  // names of domain variables
    "mom_x", "mom_y", "conti"); // names of codom. vars
// == TODO == add the equation components for the Stokes operator
xdgStokes.EquationComponents["mom_x"].Add(new Gradient_d(0)); 
xdgStokes.EquationComponents["mom_x"].Add(new Gradient_d_Interface(0)); 
xdgStokes.EquationComponents["mom_x"].Add(new Viscous(0)); 
xdgStokes.EquationComponents["mom_x"].Add(new Viscous_Interface(0)); 
xdgStokes.EquationComponents["mom_y"].Add(new Gradient_d(1)); 
xdgStokes.EquationComponents["mom_y"].Add(new Gradient_d_Interface(1)); 
xdgStokes.EquationComponents["mom_y"].Add(new Viscous(1));
xdgStokes.EquationComponents["mom_y"].Add(new Viscous_Interface(1)); 
xdgStokes.EquationComponents["conti"].Add(new Divergence());
xdgStokes.EquationComponents["conti"].Add(new Divergence_Interface());
xdgStokes.Commit();

Again, we create the matrix (now, for the Stokes operator) and check its
symmetry.

We also have to set the Reynolds number and the polynomial
degree **before** calling **ComputeMatrix** (since we are doing a
rather dirty trick by using global variables).

In [18]:
BndyMap.IsDirichletBndy  = IsDirichletBndy_Channel;
BndyMap.UDiri            = UDiri_Channel;

BlockMsrMatrix StokesMatrix_Channel = new BlockMsrMatrix(varMapChannel);
double[] StokesAffine_Channel = new double[varMapChannel.LocalLength];
xdgStokes.GetMatrixBuilder(LsTrk, varMapChannel, null, varMapChannel).ComputeMatrix(StokesMatrix_Channel, StokesAffine_Channel);

Testing the symmetry:

In [19]:
var ErrMtx1 = StokesMatrix_Channel - StokesMatrix_Channel.Transpose();
ErrMtx1.InfNorm()

1.7953077296400325E-13

## Solving the Stokes equation in the channel

The exact solution of our problem is 

$$  \vec{u}_{\text{ex}} =  (1 - y^2, 0 )^T, 
    \quad   p_{\text{ex}}    =  \frac{20}{\text{Re}} - x \frac{2}{\text{Re}} $$

and since it is polynomial we should be able to obtain it 
**exactly** in our velocity-pressure-space of degrees $(2,1)$.

First we need the corresponding right-hand-side of our problem. *BoSSS* provides us with a system 
$$
  \texttt{StokesMatrix Channel} \cdot (u,v,p) 
  + \texttt{StokesAffine Channel} = 0,
$$
so we have to multiply `StokesAffine_Channel` with $-1$ to get a 
right-hand-side.

In [20]:
double[] RHS = StokesAffine_Channel.CloneAs();
RHS.ScaleV(-1.0);

In order to store our solution, we have to create the solution DG fields:

In [21]:
XDGField u                  = new XDGField(VelBChannel, "u");
XDGField v                  = new XDGField(VelBChannel, "v");
XDGField p                  = new XDGField(PBChannel, "p");
CoordinateVector Solution_Channel   = new CoordinateVector(u, v, p);

Now we are ready to solve the linear system using a direct method:

In [22]:
StokesMatrix_Channel.Solve_Direct(Solution_Channel, RHS);

== TASK == Verify that the computed solution is actually the exact solution 

We compare to the exact solution 

In [23]:
var u_ex         = new XDGField(VelBChannel, "$u_{exact}$");
u_ex.ProjectField((double[] X) => 1.0 - X[1]*X[1]);
var v_ex         = new XDGField(VelBChannel, "$v_{exact}$");
v_ex.ProjectField((double[] X) => 0.0);
var p_ex         = new XDGField(PBChannel, "$p_{exact}$");
p_ex.GetSpeciesShadowField("A").ProjectField((double[] X) => (20.0 - X[0]*2.0)/ProblemMap.ReA);
p_ex.GetSpeciesShadowField("B").ProjectField((double[] X) => (20.0 - X[0]*2.0)/ProblemMap.ReB);
CoordinateVector Solution_Channel_ex   = new CoordinateVector(u_ex, v_ex, p_ex);

In [24]:
var check = Solution_Channel.CloneAs();
check.Acc(-1.0, Solution_Channel_ex);
foreach (var fld in check.Fields) {
    Console.WriteLine($"{fld.Identification}  L2 norm = {fld.L2Norm()}");
}

u  L2 norm = 1.8830071748812157E-14
v  L2 norm = 7.30190464803104E-15
p  L2 norm = 2.570589798826209E-14


We also verify that the evaluation of the operator and the RHS results in a zero residual.

In [25]:
XDGField momX_res                   = new XDGField(VelBChannel, "momX_residual");
XDGField momY_res                   = new XDGField(VelBChannel, "momY_residual");
XDGField conti_res                  = new XDGField(PBChannel, "conti_residual");
CoordinateVector Residual_Channel   = new CoordinateVector(momX_res, momY_res, conti_res);

In [26]:
Residual_Channel.Acc(1.0, RHS);
StokesMatrix_Channel.SpMV(-1.0, Solution_Channel, 1.0, Residual_Channel);
Residual_Channel.L2Norm()

6.592575657559547E-14

or

In [27]:
Residual_Channel.Clear();
xdgStokes.GetEvaluatorEx(LsTrk, Solution_Channel.Mapping, null, Residual_Channel.Mapping).Evaluate(1.0, 0.0, Residual_Channel);  // evaluate 
Residual_Channel.L2Norm()

7.560354085461622E-14

We export the solution to a Tecplot file, use Visit (or any other visualization software)
to inspect the solution:

In [ ]:
DGField[] plotFields = Solution_Channel.Fields
                            .Concat<DGField>(Residual_Channel.Fields).ToArray();

In [29]:
Tecplot("plots/plot_HandsOn4_ChannelFlow",  plotFields);

Writing output file plots/plot_HandsOn4_ChannelFlow...
done.
